In [5]:
import pymupdf
import re
from typing import Dict
import unicodedata
import json
from os import makedirs, path, listdir
from pathlib import Path
from PIL import Image

# Helpers

In [2]:
def date_to_en(date):
    meses = {
        "janeiro": "January",
        "fevereiro": "February",
        "março": "March",
        "marco": "March",
        "abril": "April",
        "maio": "May",
        "junho": "June",
        "julho": "July",
        "agosto": "August",
        "setembro": "September",
        "outubro": "October",
        "novembro": "November",
        "dezembro": "December"
    }
    mes = date.split()[2]
    dia = date.split()[0]
    ano = date.split()[-1]
    mes_en = meses[mes]
    date_en = f'{mes_en} {dia}, {ano}'
    return date_en

def date_to_es(date):
    meses = {
        "janeiro": "enero",
        "fevereiro": "febrero",
        "março": "marzo",
        "marco": "marzo",
        "abril": "abril",
        "maio": "mayo",
        "junho": "junio",
        "julho": "julio",
        "agosto": "agosto",
        "setembro": "septiembre",
        "outubro": "octubre",
        "novembro": "noviembre",
        "dezembro": "diciembre"
    }
    mes = date.split()[2]
    dia = date.split()[0]
    ano = date.split()[-1]
    mes_es = meses[mes]
    date_es = f'{dia} de {mes_es} de {ano}'
    return date_es

In [3]:
def extract_fields(text: str) -> Dict[str, str]:

    climatologia = re.search(r"entre\s+(\d+\s+e\s+\d+\s+mm)", text).group(1)
    min = climatologia.split()[0]
    max = climatologia.split()[2]
    observados = re.search(r"foram\s+observados\s+(\d+\s+mm)", text)
    anomalia = re.search(r"valor\s+de\s+([-\d\.]+)", text)
    classification = re.search(r"classifica\s+a\s+bacia\s+em\s+condição\s+de\s+([^\.]+)", text)
    prognostico = re.search(r"sugere\s+um\s+comportamento\s+([^\.]+)", text)
    trend = text.split("comportamento climático indica ")
    trend = trend[1].split(" ",1)[0]


    return {
        "min": min,
        "max": max,
        "observados": observados.group(1) if observados else None,
        "anomalia": anomalia.group(1) if anomalia else None,
        "classification": classification.group(1).strip() if classification else None,
        "prognostico": prognostico.group(1).strip() if prognostico else None,
        "trend": trend
    } # type: ignore


def dict_trend(trend, lang):
    d = {
        "elevacao": {
            "en": "increase",
            "es": "elevación"
        },
        "manutencao": {
            "en": "increase",
            "es": "maintenance"
        },
        "reducao": {
            "en": "decrease",
            "es": "reducción"
        }
    }
    return d[trend][lang]

In [4]:
def slugify(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^\w\s-]", "", text).strip().lower()
    text = re.sub(r"[-\s]+", "-", text)
    return text

In [5]:
def dict_classes(slugname, lang):
    d = {
        'seco': {
            "en": "dry condition",
            "es": "seco"
        },
        'tendencia-a-muito-chuvoso': {
            "en": "tendency to very rainy condition",
            "es": "tendiendo a muy lluvioso"
        },
        'tendencia-a-chuvoso': {
            "en": "tendency to rainy condition",
            "es": "tendiendo a lluvioso"
        },
        'normalidade': {
            "en": "normal contition",
            "es": "normalidad"
        },
        'tendencia-a-extremamente-seco': {
            "en": "tendency to extremely dry condition",
            "es": "tendiendo a extremadamente seco"
        },
        'tendencia-a-muito-seco': {
            "en": "tendency to very dry contition",
            "es": "tendiendo a muy seco"
        },
         'tendencia-a-seco': {
            "en": "tendency to dry contition",
            "es": "tendiendo a seco"
        },
         'muito-seco': {
            "en": "very dry condition",
            "es": "muy seco"
        },
         'chuvoso': {
            "en": "rainy",
            "es": "lluvioso"
        },
         'muito-chuvoso': {
            "en": "ver rainy",
            "es": "muy lluvioso"
        },
         'extremamente-chuvoso': {
            "en": "extremely rainy",
            "es": "extremadamente lluvioso"
        },
         'tendencia-a-extremamente-chuvoso': {
            "en": "tending to be extremely rainy",
            "es": "tendencia a ser extremadamente lluvioso"
        },
         'extremamente-seco-ou-tendencia-a-extremamente-seco': {
            "en": "extremely dry behavior or a tending to be extremely dry",
            "es": "extremadamente seco o con tendencia a ser extremadamente seco"
        },
         'extremamente-seco': {
            "en": "extremely dry",
            "es": "extremadamente seco"
        }
    }
    return d[slugname][lang]

In [6]:
def dict_trend(trend, lang):
    d = {
        "elevacao": {
            "en": "increase",
            "es": "elevación"
        },
        "manutencao": {
            "en": "increase",
            "es": "maintenance"
        },
        "reducao": {
            "en": "decrease",
            "es": "reducción"
        }
    }
    return d[trend][lang]

In [7]:
def dict_prognostico(prognostico, lang):
    d = {
        "muito-seco-ou-tendencia-a-extremamente-seco": {
            "en": "very dry behavior or a tending to be extremely dry",
            "es": "muy seco o con tendencia a ser extremadamente seco"
        },
        "muito-seco-ou-tendencia-a-muito-seco": {
            "en": "very dry behavior or a tending to be very dry",
            "es": "muy seco o con tendencia a ser muy seco"
        },
        "seco-ou-tendencia-a-muito-seco": {
            "en": "dry behavior or a tending to be very dry",
            "es": "seco o con tendencia a ser muy seco"
        },
        "chuvoso-ou-tendencia-a-chuvoso": {
            "en": "rainy behavior or a tending to be rainy",
            "es": "lluvioso o propenso a lluvioso."
        },
        "proximo-da-normalidade-ou-tendencia-a-chuvoso": {
            "en": "behavior close to normality or a tending to be rainy",
            "es": "cerca de la normalidad o con tendencia a lluvioso"
        },
        "proximo-da-normalidade-ou-tendencia-a-seco": {
            "en": "behavior close to normality or a tending to be dry",
            "es": "cerca de la normalidad o con tendencia a ser seco"
        },
         "proximo-da-normalidade": {
            "en": "behavior close to normality",
            "es": "cerca de la normalidad"
        },
         "seco-ou-tendencia-a-seco": {
            "en": "dry behavior or a tending to be dry",
            "es": "seco o con tendencia a ser seco"
        },
         'chuvoso-ou-tendencia-a-muito-chuvoso': {
            "en": "rainy behavior or a tending to be very rainy",
            "es": "lluvioso o con tendencia a ser muy lluvioso"
        },
         'muito-chuvoso-ou-tendencia-a-muito-chuvoso': {
            "en": "very rainy behavior or a tending to be very rainy",
            "es": "muy lluvioso o con tendencia a muy lluvioso"
        },
         'muito-chuvoso-ou-tendencia-a-extremamente-chuvoso': {
            "en": "very rainy behavior or a tending to be extremely rainy",
            "es": "muy seco o con tendencia a ser extremadamente seco"
        },
         'extremamente-chuvoso-ou-tendencia-a-extremamente-chuvoso': {
            "en": "extremely rainy behavior or a tending to be extremely rainy",
            "es": "extremadamente lluvioso o con tendencia a extremadamente lluvioso"
        },
         'extremamente-seco-ou-tendencia-a-extremamente-seco': {
            "en": "extremely dry behavior or a tending to be extremely dry",
            "es": "extremadamente seco o con tendencia a ser extremadamente seco"
        }
    }
    return d[prognostico][lang]  

# Meta

In [8]:
def get_meta(doc, bulletin_dict):
    # Page 1
    page = doc.load_page(0)
    text = page.get_text()
    text_lines = text.splitlines()
    number = text_lines[-1].split()[3]
    date = " ".join(text_lines[-1].split()[5:])

    meta = {
        "meta": {
        "pt": {
            "doi": f'10.61818/029106{number}',
            "issn": "2965-0291",
            "volume": "06",
            "date": date
        },
        "en": {
            "doi": f'10.61818/785704{number}',
            "issn": "2965-7857",
            "volume": "04",
            "date": date_to_en(date)
        },
        "es": {
            "doi": f'10.61818/770904{number}',
            "issn": "2965-7709",
            "volume": "04",
            "date": date_to_es(date)
        },
         "number": number
    }
        }
    bulletin_dict.update(meta)
    return bulletin_dict


# get_current_conditions

In [9]:
def get_current_conditions(pathPT, pathEN, pathES, bulletin_dict):
    doc = pymupdf.open(pathPT)     
    page = doc.load_page(3)
    textPT = page.get_text()
    textPT = re.sub(r'\n+', ' ', textPT)
    textPT = re.sub(r'\s+', ' ', textPT)
    textPT = textPT.split("Condições atuais ")[1]
    # textPT = textPT.split("CODAM Página 1 ")[1]
    textPT = textPT.split("1 Abacaxis ")[0].rstrip()
    # EN
    doc = pymupdf.open(pathEN)     
    page = doc.load_page(3)
    textEN = page.get_text()
    textEN = re.sub(r'\n+', ' ', textEN)
    textEN = re.sub(r'\s+', ' ', textEN)
    textEN = textEN.split("Current conditions ")[1]
    textEN = textEN.split("1 Abacaxis ")[0].rstrip()
    # ES
    doc = pymupdf.open(pathES)     
    page = doc.load_page(3)
    textES = page.get_text()
    textES = re.sub(r'\n+', ' ', textES)
    textES = re.sub(r'\s+', ' ', textES)
    textES = textES.split("Condiciones actuales ")[1]
    textES = textES.split("1 Abacaxis ")[0].rstrip()
    
    current_conditions = {
        "pt": textPT,
        "en": textEN,
        "es": textES
    }
    
    bulletin_dict['current_conditions'] = current_conditions
    return bulletin_dict 

# get_analysis

In [10]:
def get_analysis(doc, bulletin_dict):
    bacias = ['Bacia do Rio Branco', 'Bacia do Rio Negro','Bacia do Rio Marañon', 'Bacia do Rio Ucayali', 'Bacia do Rio Napo', 
'Curso principal do Rio Amazonas (Peru)','Bacia do Rio Javari','Bacia do Rio Içá (Putumayo)','Bacia do Rio Jutaí','Bacia do Rio Juruá',
'Bacia do Rio Japurá (Caquetá)','Bacia do Rio Tefé','Bacia do Rio Coari','Bacia do Rio Purus','Curso principal do Rio Solimões',
'Bacia dos rios Beni e Madre de Dios','Bacia do Rio Mamoré','Bacia do Rio Guaporé (Iténez)','Bacia do Rio Ji-Paraná','Bacia do Rio Aripuanã',
'Bacia do Rio Madeira','Bacias da margem esquerda do Rio Amazonas (Amazonas)','Bacia do Rio Abacaxis','Bacia do Rio Juruena','Bacia do Rio Teles Pires',
'Bacia do Rio Tapajós','Bacias da margem esquerda do Rio Amazonas (noroeste do Pará)','Bacia do Rio Curuá Una','Bacias da margem esquerda do Rio Amazonas (nordeste do PA)',
'Bacia do Rio Iriri','Bacia do Rio Xingu','Curso principal do Rio Amazonas (Brasil)']
    analysis = []
    texts = [doc.load_page(pg).get_text() for pg in range(4,15)]
    text = " ".join(texts)
    text = re.sub(r"\s*\n\s*", " ", text)
    text = re.sub(r"\s{2,}", " ", text).strip()
    for i, nome in enumerate(bacias):
        nome_regex = re.escape(nome)
        start_match = re.search(nome_regex, text)
        start = start_match.end()
        if i < len(bacias) - 1:
            next_nome = re.escape(bacias[i + 1])
            next_match = re.search(next_nome, text)
            end = next_match.start() if next_match else len(text)
        else:
            end = len(text)
        basin_text = text[start:end].strip()
        fields = extract_fields(basin_text)
        slug_name = slugify(nome)
        classification = fields["classification"]
        slug_classification = slugify(classification)
        # trend
        trend = fields["trend"]
        slug_trend = slugify(trend)
        # prognostico
        prognostico = fields["prognostico"]
        slug_prognostico = slugify(prognostico)

        
        bacia = {
                "id": slug_name,
                "name": nome,
                "min": fields["min"],
                "max": fields["max"],
                "observados": fields["observados"],
                "anomalia": fields["anomalia"],
                "i18n": {
                    "pt": {
                        "classification": classification,
                        "trend": trend,
                        "prognostico": prognostico
                    },
                    "en": {
                        "classification": dict_classes(slug_classification, "en"),
                        "trend": dict_trend(slug_trend, "en"),
                        "prognostico": dict_prognostico(slug_prognostico, "en")
                    },
                    "es": {
                        "classification": dict_classes(slug_classification, "es"),
                        "trend": dict_trend(slug_trend, "es"),
                        "prognostico": dict_prognostico(slug_prognostico, "es")
                    }
                }
            }
        # print(bacia)
        analysis.append(bacia)
    bulletin_dict['analysis'] = analysis
    
    return bulletin_dict

# get_multimodel

In [11]:
def get_multimodel(pathPT, pathEN, pathES, bulletin_dict):
    multimodel = {
        "pt": {},
        "en": {},
        "es": {}
    }
    # pt
    doc = pymupdf.open(pathPT)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "A Figura acima," in text:
            seven_days = text.replace("\n", "")
            multimodel['pt']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "A Figura acima," in text:
            fourteen_days = text.replace("\n", "")
            multimodel['pt']['fourteen_days'] = fourteen_days
            break
    # en
    doc = pymupdf.open(pathEN)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "The figure above"in text:
            seven_days = text.replace("\n", "")
            multimodel['en']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "The figure above" in text:
            fourteen_days = text.replace("\n", "")
            multimodel['en']['fourteen_days'] = fourteen_days
            break
    # es
    doc = pymupdf.open(pathES)
    page = doc.load_page(15)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "La figura de arriba"in text:
            seven_days = text.replace("\n", "")
            multimodel['es']['seven_days'] = seven_days
            break
    page = doc.load_page(16)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        text = re.sub(r'\s+', ' ', text).strip()
        if "La figura de arriba" in text:
            fourteen_days = text.replace("\n", "")
            multimodel['es']['fourteen_days'] = fourteen_days
            break

    bulletin_dict['multimodel'] = multimodel


    return bulletin_dict

# Text

In [12]:
def get_text(pathPT, pathEN, pathES, yyyy, mmdd, outputPath):
    bulletin_dict = {}
    doc = pymupdf.open(pathPT)
    bulletin_dict = get_meta(doc, bulletin_dict) 
    bulletin_dict = get_current_conditions(pathPT, pathEN, pathES, bulletin_dict)
    bulletin_dict = get_analysis(doc, bulletin_dict)
    bulletin_dict = get_multimodel(pathPT, pathEN, pathES, bulletin_dict)
    with open(f'{outputPath}/{mmdd}.json', 'w') as f:
        json.dump(bulletin_dict, f, ensure_ascii=False, indent=3)
    return bulletin_dict   

In [13]:
pathPT = 'current/BHA_PT_20260408.pdf'
pathEN = 'current/BHA_EN_20260408.pdf'
pathES = 'current/BHA_ES_20260408.pdf'
yyyy = '2026'
mmdd = '0408'
outputPath = 'test'
get_text(pathPT, pathEN, pathES, yyyy, mmdd, outputPath)

{'meta': {'pt': {'doi': '10.61818/02910614',
   'issn': '2965-0291',
   'volume': '06',
   'date': '8 de abril de 2026'},
  'en': {'doi': '10.61818/78570414',
   'issn': '2965-7857',
   'volume': '04',
   'date': 'April 8, 2026'},
  'es': {'doi': '10.61818/77090414',
   'issn': '2965-7709',
   'volume': '04',
   'date': '8 de abril de 2026'},
  'number': '14'},
 'current_conditions': {'pt': 'Mapas das condições observadas de precipitação e gráficos individuais por bacias são produzidos a partir dos dados MERGE/GPM gerados pelo INPE/CPTEC, considerando como climatologia para período de 2000 a 2025. Entre os dias 10 de março e 8 de abril de 2026, chuvas abaixo da climatologia caracterizaram com déficit de precipitação o curso principal do Rio Amazonas em territórios brasileiro e peruano, bacias hidrográficas dos rios Abacaxis, Aripuanã, Coari, Curuá Una, Içá, Iriri, Japurá, Javari, Ji-Paraná, Juruá, Juruena, Jutaí, Madeira, bacias da margem esquerda do Rio Amazonas no nordeste do Estado 

# Imagem

# img_analysis

In [14]:
def img_analysis(doc, output_path):
    bacias = ['Bacia do Rio Branco', 'Bacia do Rio Negro','Bacia do Rio Marañon', 'Bacia do Rio Ucayali', 'Bacia do Rio Napo', 
    'Curso principal do Rio Amazonas (Peru)','Bacia do Rio Javari','Bacia do Rio Içá (Putumayo)','Bacia do Rio Jutaí','Bacia do Rio Juruá',
    'Bacia do Rio Japurá (Caquetá)','Bacia do Rio Tefé','Bacia do Rio Coari','Bacia do Rio Purus','Curso principal do Rio Solimões',
    'Bacia dos rios Beni e Madre de Dios','Bacia do Rio Mamoré','Bacia do Rio Guaporé (Iténez)','Bacia do Rio Ji-Paraná','Bacia do Rio Aripuanã',
    'Bacia do Rio Madeira','Bacias da margem esquerda do Rio Amazonas (Amazonas)','Bacia do Rio Abacaxis','Bacia do Rio Juruena','Bacia do Rio Teles Pires',
    'Bacia do Rio Tapajós','Bacias da margem esquerda do Rio Amazonas (noroeste do Pará)','Bacia do Rio Curuá Una','Bacias da margem esquerda do Rio Amazonas (nordeste do PA)',
    'Bacia do Rio Iriri','Bacia do Rio Xingu','Curso principal do Rio Amazonas (Brasil)']
    
    path = f"{output_path}/analysis"
    images = []
    for b in bacias:
        slug_name = slugify(b)
        images.append(f'{slug_name}-acc.png')
        images.append(f'{slug_name}-ano.png')
    c = 0
    for i in range(4, 15):
        page = doc.load_page(i)
        d = page.get_text("dict")
        blocks = d["blocks"] 
        imgblocks = [b for b in blocks if b["type"] == 1]
        for i in imgblocks:
            if i['bbox'][0] < 100.0:
                img = i['image']
                img_name = images[c]
                with open(f'{path}/{img_name}', "wb") as f:
                                f.write(img)
                c += 1

# img_current_conditions

In [15]:
def img_current_conditions(doc, output_path):

    path = f"{output_path}/current_conditions/"
    page = doc.load_page(3)
    # maps
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    for i in imgblocks:
        if i['height'] > 400:
            img = i['image'] 
            break
    map = f'{path}map_current_conditions.png'
    with open(map, "wb") as f:
                    f.write(img) # type: ignore

    
    
    # Tabels
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if "1\nAbacaxis" in text:
            break

    top = y0 - 5 # type: ignore
    left = x0 - 15 # type: ignore
    right = x1 + 45 # type: ignore
    bottom = y1 + 5 # type: ignore
    rect = pymupdf.Rect(left, top, right, bottom)
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}table_current_conditions.png')

# img_multimodel

In [16]:
def img_multimodel(doc, output_path):
    path = f"{output_path}/multimodel"
    page = doc.load_page(15)
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    for i in imgblocks:
        if i['height'] > 400:
            img = i
            break
    left, top, right, bottom = img['bbox'] # type: ignore
    right += 100
    rect = pymupdf.Rect(left, top, right, bottom)
    zoom = 3
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}/seven_days.png')
    
    # fourteen_days
    page = doc.load_page(16)
    d = page.get_text("dict")
    blocks = d["blocks"] 
    imgblocks = [b for b in blocks if b["type"] == 1]
    for i in imgblocks:
        if i['height'] > 400:
            img = i
            break
    left, top, right, bottom = img['bbox'] # type: ignore
    right += 100
    rect = pymupdf.Rect(left, top, right, bottom)
    zoom = 3
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}/fourteen_days.png')

# img_reference

In [17]:
def img_reference(doc, output_path):
    path = f"{output_path}/reference"
    page = doc.load_page(17)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if 'Abacaxis' in text:
            break
    rect = pymupdf.Rect(x0, y0, x1, y1) # type: ignore
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}/reference_table.png')

# img_categorization

In [18]:
def img_categorization(doc, output_path):
    path = f"{output_path}/categorization"
    page = doc.load_page(18)
    blocks = page.get_text("blocks")
    for x0, y0, x1, y1, text, *_ in blocks:
        if 'Abacaxis' in text:
            break
    rect = pymupdf.Rect(x0, y0, x1, y1) # type: ignore
    zoom = 3 
    mat = pymupdf.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
    pix.save(f'{path}/table.png')

# img_anomaly

In [19]:
def img_anomaly(doc, output_path):
    c = 1
    path = f"{output_path}/anomaly"
    for p in range(19,23):
        page = doc.load_page(p)
        d = page.get_text("dict")
        blocks = d["blocks"] 
        imgblocks = [b for b in blocks if b["type"] == 1]
        for i in imgblocks:
            if i['height'] > 250:
                bbox = i['bbox']
                rect = pymupdf.Rect(bbox)
                zoom = 3
                mat = pymupdf.Matrix(zoom, zoom)
                pix = page.get_pixmap(matrix=mat, clip=rect, alpha=False)
                pix.save(f'{path}/bacia_{c}.png')
                c += 1

# get_images

In [20]:
def get_images(pathPT, output_path):
    
    doc = pymupdf.open(pathPT)
    img_current_conditions(doc, output_path)
    img_analysis(doc, output_path)
    img_multimodel(doc, output_path)
    img_reference(doc, output_path)
    img_categorization(doc, output_path)
    img_anomaly(doc, output_path)

In [21]:
output_path = 'test/images'
makedirs(output_path, exist_ok=True)
makedirs(f'{output_path}/current_conditions', exist_ok=True)
makedirs(f'{output_path}/analysis', exist_ok=True)
makedirs(f'{output_path}/multimodel', exist_ok=True)
makedirs(f'{output_path}/reference', exist_ok=True)
makedirs(f'{output_path}/categorization', exist_ok=True)
makedirs(f'{output_path}/anomaly', exist_ok=True)
get_images(pathPT, output_path)

In [7]:
path_img = Path('test/0401')
out_img_path = Path('test/webp')
makedirs(out_img_path, exist_ok=True)
for section in listdir(path_img):
    path_section = Path(path_img / section)
    imgs = listdir(path_section)
    path_out_section = f'test/images_webp/{section}'
    makedirs(path_out_section, exist_ok=True)
    for img in imgs:
        img_path = Path(path_section / img)
        name = img.split('.')[0]
        out_img_path = f'{path_out_section}/{name}.webp'
        with Image.open(img_path) as im:
            img = im.convert("RGB")
            img.save(out_img_path, "WEBP", quality=50)
            im.save(img_path.with_suffix('.jpg'), 'JPEG')
    
    

In [ ]:
path_year = Path('bk/2026')
total_size = 0
out_dir = Path('images/quality_50')
for edition in listdir(path_year):
    path_edition = listdir(path_year / edition)
    path_out_edition = out_dir / edition
    makedirs(path_out_edition, exist_ok=True)
    for section in path_edition:
        path_section = Path(path_year / edition / section)
        if path.isdir(path_section):
            path_out_section = path_out_edition / section
            makedirs(path_out_section, exist_ok=True)
            imgs = listdir(path_section)
            for img in imgs:
                name = img.split('.')[0]
                img_path = Path(path_section / img)
                out_img_path = path_out_section / f'{name}.webp'
                print(out_img_path)
                # orignal_size = round(path.getsize(img_path) / 1024)
                # total_size += orignal_size
                with Image.open(img_path) as img:
                    img = img.convert("RGB")
                    img.save(out_img_path, "WEBP", quality=50)

In [12]:
old_img = '/home/inacio/clima-amazonia/public/issues/2026/mar/0304.png'
new_img = '/home/inacio/clima-amazonia/public/issues/2026/mar/0304.webp'

In [13]:
with Image.open(old_img) as img:
    img = img.convert("RGB")
    img.save(new_img, "WEBP", quality=50)

In [67]:
with open('test/0408.json') as f:
    data = json.load(f)
    images = data['images']
    for k in images.keys():   
        for j in images[k].keys():   
            url = images[k][j]
            x = url.split('0325')[1]   
            x = "".join(x.split('.')[:-1])
            new_url = f'https://climamazonia.shop/2026/0408{x}.webp'   
            images[k][j] = new_url
with open('/home/inacio/clima-amazonia/data/boletim/2026/0408.json', 'w') as f:
    json.dump(data, f, ensure_ascii=False, indent=3)

In [53]:
images[k]

{'table': 'https://res.cloudinary.com/dveg6vhbi/image/upload/v1774535110/clima-amazonia/2026/0325/categorization/table.png'}

# Uploads

In [1]:
! pip install qrcode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 552.6 kB/s eta 0:00:00a 0:00:01


In [2]:
import qrcode

In [3]:
data = "https://climaamazonia.org/"

img = qrcode.make(data)
img.save("qrcode.png")

# DOI

In [1]:
from datetime import datetime
import secrets

In [4]:
token = secrets.token_hex(12)
token


'58684b4b38c108afd5afa32b'

In [5]:
datetime.now().strftime("%Y%m%d%H%M%S%f")[:-3]

'20260416084630325'